In [1]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

Found existing installation: keras 3.10.0
Uninstalling keras-3.10.0:
  Successfully uninstalled keras-3.10.0
Found existing installation: matplotlib 3.10.0
Uninstalling matplotlib-3.10.0:
  Successfully uninstalled matplotlib-3.10.0
Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
Found existing installation: tensorflow 2.19.0
Uninstalling tensorflow-2.19.0:
  Successfully uninstalled tensorflow-2.19.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
warnings.simplefilter('ignore')

In [3]:
import os
import sys
import subprocess

In [11]:
!rm -rf /kaggle/tmp/setup

In [17]:
def set_env(input_archive, temp_dir):

    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir, exist_ok=True)
        subprocess.run(['tar', '-xzf', input_archive, '-C', temp_dir], check=True)
    
    subprocess.run([
        sys.executable, '-m', 'pip', 'install',
        '--no-index',
        '--find-links', f'{temp_dir}/wheels',
        '--upgrade',
        '--force-reinstall',
        '--no-deps',
        'vllm',
        'transformers',
    ], check=True)
    
    subprocess.run([
        sys.executable, '-m', 'pip', 'install',
        '--no-index',
        '--find-links', f'{temp_dir}/wheels',
        'vllm',
        'openai',
        'tiktoken',
        'transformers',
        'accelerate',
        'polars',
    ], check=True)

In [18]:
# Run this as the FIRST cell in your Kaggle notebook
# It will find every input path you need

import os

print("=" * 70)
print("ALL KAGGLE INPUTS")
print("=" * 70)

# 1. Show top-level input structure
print("\n📁 /kaggle/input/ top-level:")
if os.path.exists('/kaggle/input'):
    for item in sorted(os.listdir('/kaggle/input')):
        full = os.path.join('/kaggle/input', item)
        if os.path.isdir(full):
            print(f"  📂 {item}/")
        else:
            print(f"  📄 {item}")

# 2. Find ALL config.json files (model locations)
print("\n" + "=" * 70)
print("🤖 MODEL LOCATIONS (config.json)")
print("=" * 70)
found_models = []
for root, dirs, files in os.walk('/kaggle/input'):
    if 'config.json' in files:
        found_models.append(root)
        print(f"  {root}")
        try:
            import json
            with open(os.path.join(root, 'config.json')) as f:
                cfg = json.load(f)
            model_type = cfg.get('model_type', 'unknown')
            archs = cfg.get('architectures', ['unknown'])
            print(f"    model_type: {model_type}")
            print(f"    architectures: {archs}")
        except:
            pass

if not found_models:
    print("  ⚠️  No models found! Add a model via '+ Add Input' → 'Models' tab")

# 3. Find test.csv / reference.csv (competition data)
print("\n" + "=" * 70)
print("📊 COMPETITION DATA (csv files)")
print("=" * 70)
found_csv = False
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.csv'):
            found_csv = True
            print(f"  {os.path.join(root, f)}")

if not found_csv:
    print("  ⚠️  No CSV files found! Add competition data as input")

# 4. Find wheels.tar.gz (utils)
print("\n" + "=" * 70)
print("📦 WHEELS / UTILS (tar.gz files)")
print("=" * 70)
found_tar = False
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.tar.gz') or f.endswith('.tgz'):
            found_tar = True
            size_mb = os.path.getsize(os.path.join(root, f)) / (1024*1024)
            print(f"  {os.path.join(root, f)}  ({size_mb:.1f} MB)")

if not found_tar:
    print("  ⚠️  No tar.gz found! Upload your wheels dataset")

# 5. Find any .safetensors or .bin (model weight files)
print("\n" + "=" * 70)
print("⚖️  MODEL WEIGHT FILES")
print("=" * 70)
weight_dirs = set()
total_weight_size = 0
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.safetensors') or f.endswith('.bin'):
            weight_dirs.add(root)
            total_weight_size += os.path.getsize(os.path.join(root, f))

for d in sorted(weight_dirs):
    n_files = len([f for f in os.listdir(d) if f.endswith('.safetensors') or f.endswith('.bin')])
    print(f"  {d}  ({n_files} weight files)")

if weight_dirs:
    print(f"\n  Total weight size: {total_weight_size / 1e9:.2f} GB")
else:
    print("  ⚠️  No weight files found!")

# 6. Summary: what to put in CFG
print("\n" + "=" * 70)
print("📋 COPY THESE INTO YOUR CFG CLASS")
print("=" * 70)

if found_models:
    print(f"\n  model_path = '{found_models[0]}'")
else:
    print("\n  model_path = ???  # No model found — add it as input!")

for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f == 'wheels.tar.gz':
            print(f"  wheels_path = '{os.path.join(root, f)}'")
            break

for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f in ('test.csv', 'reference.csv'):
            print(f"  data_path = '{os.path.join(root, f)}'")

print("\n" + "=" * 70)
print("DONE")
print("=" * 70)

ALL KAGGLE INPUTS

📁 /kaggle/input/ top-level:
  📂 competitions/
  📂 models/
  📂 notebooks/

🤖 MODEL LOCATIONS (config.json)
  /kaggle/input/models/google/gemma-4/transformers/gemma-4-26b-a4b/1
    model_type: gemma4
    architectures: ['Gemma4ForConditionalGeneration']

📊 COMPETITION DATA (csv files)
  /kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/reference.csv
  /kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/sample_submission.csv
  /kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv

📦 WHEELS / UTILS (tar.gz files)
  /kaggle/input/notebooks/pranshusharmaa/notebook23416a77c4/wheels.tar.gz  (4676.6 MB)

⚖️  MODEL WEIGHT FILES
  /kaggle/input/models/google/gemma-4/transformers/gemma-4-26b-a4b/1  (2 weight files)

  Total weight size: 51.61 GB

📋 COPY THESE INTO YOUR CFG CLASS

  model_path = '/kaggle/input/models/google/gemma-4/transformers/gemma-4-26b-a4b/1'
  wheels_path = '/kaggle/input/notebooks/pranshusharmaa/no

In [20]:
!pip install --no-index --find-links /kaggle/tmp/setup/wheels --upgrade --force-reinstall --no-deps vllm-0.19.0*.whl 2>/dev/null || \
 echo "0.19.0 wheel not found — checking what's available:" && \
 ls /kaggle/tmp/setup/wheels/vllm*

0.19.0 wheel not found — checking what's available:
/kaggle/tmp/setup/wheels/vllm-0.11.0-cp38-abi3-manylinux1_x86_64.whl


In [19]:
set_env(
    input_archive='/kaggle/input/notebooks/pranshusharmaa/notebook23416a77c4/wheels.tar.gz', 
    temp_dir='/kaggle/tmp/setup'
)

Looking in links: /kaggle/tmp/setup/wheels
Processing /kaggle/tmp/setup/wheels/vllm-0.11.0-cp38-abi3-manylinux1_x86_64.whl
Processing /kaggle/tmp/setup/wheels/transformers-5.5.3-py3-none-any.whl
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
Looking in links: /kaggle/tmp/setup/wheels
Processing /kaggle/tmp/setup/wheels/prometheus_fastapi_instrumentator-7.1.0-py3-none-any.whl (from vllm)
Processing /kaggle/tmp/setup/wheels/lm_format_enforcer-0.11.3-py3-none-any.whl (from vllm)
Processing /kaggle/tmp/setup/wheels/llguidance-0.7.30-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (from vllm)
Processing /kaggle/tmp/setup/wheels/outlines_core-0.2.11-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (from vllm)
Processing /kaggle/tmp/setup/wheels/diskcache-5.6.3-py3-none-any.whl (from vllm)
Processing /kaggle/tmp/setup/wheels/lark-1.2.2-py3-none-an

ERROR: Operation cancelled by user


KeyboardInterrupt: 

In [ ]:
subprocess.run(['ls', '/kaggle/tmp/setup/tiktoken_encodings'])

In [ ]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
os.environ['TIKTOKEN_ENCODINGS_BASE'] = '/kaggle/tmp/setup/tiktoken_encodings'

In [ ]:
import gc
import re
import math
import time
import queue
import threading
import contextlib
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor

import pandas as pd
import polars as pl

from openai import OpenAI
from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server

In [ ]:
class CFG:

    # ── System Prompt (Gemma 4 thinking mode) ──
    system_prompt = (
        '<|think|>'
        'You are a world-class International Mathematical Olympiad (IMO) competitor. '
        'Use Python to compute and verify your results. '
        'The final answer must be a non-negative integer between 0 and 99999. '
        'You must place the final integer answer inside \\boxed{}.'
    )
    
    tool_prompt = (
        'Use this tool to execute Python code. '
        'The environment is a stateful Jupyter notebook. '
        'You must use print() to output results.'
    )

    preference_prompt = (
        'Use `math`, `numpy`,`sympy`,`itertools` and `collections` to solve the problem.'
    )

    # ── Model ──
    served_model_name = 'gemma4'
    model_path = '/kaggle/input/models/google/gemma-4/transformers/gemma-4-26b-a4b/1'
    kv_cache_dtype = 'auto'
    dtype = 'bfloat16'

    # ── Timing ──
    high_problem_timeout = 900
    base_problem_timeout = 270
    notebook_limit = 17400
    server_timeout = 180
    session_timeout = 960
    jupyter_timeout = 10
    sandbox_timeout = 5

    # ── Serving ──
    context_tokens = 65536
    batch_size = 256
    workers = 16
    turns = 128
    seed = 42
    gpu_memory_utilization = 0.95

    # ── Sampling (Google recommended for Gemma 4) ──
    temperature = 1.0
    top_p = 0.95
    top_k = 64

    # ── Branch Strategy (adapted for Gemma 4) ──
    n_tir_branches = 10       # Code-enabled branches
    n_cot_branches = 4       # Pure reasoning branches (no code)
    attempts = 12            # Total = n_tir + n_cot
    early_stop = 5           # Stop if 5 branches agree
    rescue_branches = 6      # Extra TIR branches on weak consensus

    # ── GenSelect ──
    use_genselect = True
    genselect_passes = 4     # Selector passes at temp 0.3

In [ ]:
set_seed(CFG.seed)

In [ ]:
class Gemma4Template:

    def __init__(self):
        pass

    def apply_chat_template(
        self, 
        system_prompt: str, 
        user_prompt: str, 
        tool_instruction: str = '',
    ) -> list[dict]:

        full_system = system_prompt
        if tool_instruction:
            full_system += '\n\n' + tool_instruction

        return [
            {'role': 'system', 'content': full_system},
            {'role': 'user', 'content': user_prompt},
        ]

In [ ]:
class AIMO3Sandbox:

    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list[int]:

        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count

            return ports

    def __init__(self, timeout: float):

        self._default_timeout = timeout
        self._owns_kernel = False
        self._client = None
        self._km = None
        
        ports = self._get_next_ports(5)

        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT'] = '0'
        env['JUPYTER_PLATFORM_DIRS'] = '1'
        env['PYTHONWARNINGS'] = 'ignore'
        env['MPLBACKEND'] = 'Agg'

        self._km = KernelManager()
        self._km.shell_port = ports[0]
        self._km.iopub_port = ports[1]
        self._km.stdin_port = ports[2]
        self._km.hb_port = ports[3]
        self._km.control_port = ports[4]

        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])

        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True

        self.execute(
            'import math\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import numpy as np\n'
        )

    def _format_error(self, traceback: list[str]) -> str:

        clean_lines = []

        for frame in traceback:
            clean_frame = re.sub(r'\x1b\[[0-9;]*m', '', frame)

            if 'File "' in clean_frame and 'ipython-input' not in clean_frame:
                continue

            clean_lines.append(clean_frame)

        return ''.join(clean_lines)

    def execute(self, code: str, timeout: float | None = None) -> str:

        client = self._client
        effective_timeout = timeout or self._default_timeout
        
        msg_id = client.execute(
            code, 
            store_history=True, 
            allow_stdin=False, 
            stop_on_error=False
        )

        stdout_parts = []
        stderr_parts = []
        
        start_time = time.time()

        while True:
            elapsed = time.time() - start_time

            if elapsed > effective_timeout:
                self._km.interrupt_kernel()

                return f'[ERROR] Execution timed out after {effective_timeout} seconds'

            try:
                msg = client.get_iopub_msg(timeout=1.0)

            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != msg_id:
                continue

            msg_type = msg.get('msg_type')
            content = msg.get('content', {})

            if msg_type == 'stream':
                text = content.get('text', '')

                if content.get('name') == 'stdout':
                    stdout_parts.append(text)

                else:
                    stderr_parts.append(text)

            elif msg_type == 'error':
                traceback_list = content.get('traceback', [])

                stderr_parts.append(self._format_error(traceback_list))

            elif msg_type in {'execute_result', 'display_data'}:
                data = content.get('data', {})
                text = data.get('text/plain')

                if text:
                    stdout_parts.append(text if text.endswith('\n') else f'{text}\n')

            elif msg_type == 'status':
                if content.get('execution_state') == 'idle':
                    break

        stdout = ''.join(stdout_parts)
        stderr = ''.join(stderr_parts)

        if stderr:
            return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr

        return stdout if stdout.strip() else '[WARN] No output. Use print() to see results.'

    def close(self):

        with contextlib.suppress(Exception):
            if self._client:
                self._client.stop_channels()

        if self._owns_kernel and self._km is not None:
            with contextlib.suppress(Exception):
                self._km.shutdown_kernel(now=True)

            with contextlib.suppress(Exception):
                self._km.cleanup_resources()

    def reset(self):

        self.execute('%reset -f')
        self.execute('import gc; gc.collect()')

        self.execute(
            'import math\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import numpy as np\n'
        )

    def __del__(self):

        self.close()

In [ ]:
class Gemma4Tool:

    def __init__(self, local_jupyter_timeout: float, tool_prompt: str, sandbox=None):
        self._local_jupyter_timeout = local_jupyter_timeout
        self._tool_prompt = tool_prompt
        self._jupyter_session = sandbox
        self._owns_session = sandbox is None
        self._execution_lock = threading.Lock()
        self._init_lock = threading.Lock()

    def _ensure_session(self):
        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(timeout=self._local_jupyter_timeout)

    def _ensure_last_print(self, code: str) -> str:
        lines = code.strip().split('\n')
        if not lines:
            return code
        last_line = lines[-1].strip()
        if 'print' in last_line or 'import' in last_line:
            return code
        if not last_line or last_line.startswith('#'):
            return code
        lines[-1] = 'print(' + last_line + ')'
        return '\n'.join(lines)

    @property
    def instruction(self) -> str:
        return self._tool_prompt

    def execute_code(self, code: str) -> dict:
        self._ensure_session()
        final_script = self._ensure_last_print(code)
        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(final_script)
            except TimeoutError as exc:
                output = f'[ERROR] {exc}'
        is_error = (
            output.startswith('[ERROR]') or 
            'Traceback' in output or 
            'Error:' in output
        )
        return {'output': output, 'is_error': is_error}

    def close(self):
        if self._jupyter_session is not None:
            if self._owns_session:
                self._jupyter_session.close()
            self._jupyter_session = None

    def __del__(self):
        self.close()

In [ ]:
class Gemma4AdaptedSolver:
    """Gemma 4 solver adapted from AIMO-2 winning strategies.
    
    Key adaptations for Gemma 4 behavior:
    1. Mixed branch families: TIR (code+think) + CoT (think-only)
    2. Two-temperature: 1.0 for branches, 0.3 for GenSelect
    3. Smart GenSelect: runs on any non-unanimous result
    4. Hardened answer extraction: scans thinking blocks too
    5. Adaptive branching: easy→save time, hard→rescue+GenSelect
    """

    CODE_BLOCK_RE = re.compile(r'```(?:python)?\s*\n(.*?)```', re.DOTALL)
    THINK_RE = re.compile(r'<\|channel>thought\n.*?<channel\|>', re.DOTALL)
    
    # Hardened answer extraction — multiple patterns
    BOXED_RE = re.compile(r'\\boxed\s*\{\s*([0-9,]+)\s*\}')
    ANSWER_IS_RE = re.compile(r'(?:final answer|answer is|answer:)\s*(\d+)', re.IGNORECASE)
    THEREFORE_RE = re.compile(r'(?:therefore|thus|hence|so)\s*,?\s*(?:the answer is\s*)?(\d+)', re.IGNORECASE)

    def __init__(self, cfg, port: int = 8000):

        self.cfg = cfg
        self.port = port
        self.base_url = f'http://0.0.0.0:{port}/v1'
        self.api_key = 'sk-local'

        self._preload_model_weights()
        self.server_process = self._start_server()

        self.client = OpenAI(
            base_url=self.base_url,
            api_key=self.api_key,
            timeout=self.cfg.session_timeout
        )

        self._wait_for_server()
        self._initialize_kernels()

        self.notebook_start_time = time.time()
        self.problems_remaining = 50
        
        # Ablation tracking
        self.problem_logs = []

    def _preload_model_weights(self) -> None:

        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        start_time = time.time()

        files_to_load = []
        total_size = 0

        for root, _, files in os.walk(self.cfg.model_path):
            for file_name in files:
                file_path = os.path.join(root, file_name)
                if os.path.isfile(file_path):
                    files_to_load.append(file_path)
                    total_size += os.path.getsize(file_path)

        def _read_file(path: str) -> None:
            with open(path, 'rb') as file_object:
                while file_object.read(1024 * 1024 * 1024):
                    pass

        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            list(executor.map(_read_file, files_to_load))

        elapsed = time.time() - start_time
        print(f'Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')

    def _start_server(self) -> subprocess.Popen:

        cmd = [
            sys.executable,
            '-m', 'vllm.entrypoints.openai.api_server',
            '--seed', str(self.cfg.seed),
            '--model', self.cfg.model_path,
            '--served-model-name', self.cfg.served_model_name,
            '--tensor-parallel-size', '1',
            '--max-num-seqs', str(self.cfg.batch_size),
            '--gpu-memory-utilization', str(self.cfg.gpu_memory_utilization),
            '--host', '0.0.0.0',
            '--port', str(self.port),
            '--dtype', self.cfg.dtype,
            '--kv-cache-dtype', self.cfg.kv_cache_dtype,
            '--max-model-len', str(self.cfg.context_tokens),
            '--enable-prefix-caching',
        ]

        self.log_file = open('vllm_server.log', 'w')

        return subprocess.Popen(
            cmd,
            stdout=self.log_file,
            stderr=subprocess.STDOUT,
            start_new_session=True
        )

    def _wait_for_server(self):

        print('Waiting for vLLM server...')
        start_time = time.time()

        for _ in range(self.cfg.server_timeout):
            return_code = self.server_process.poll()
            if return_code is not None:
                self.log_file.flush()
                with open('vllm_server.log', 'r') as log_file:
                    logs = log_file.read()
                raise RuntimeError(f'Server died with code {return_code}. Full logs:\n{logs}\n')
            try:
                self.client.models.list()
                elapsed = time.time() - start_time
                print(f'Server is ready (took {elapsed:.2f} seconds).\n')
                return
            except Exception:
                time.sleep(1)

        raise RuntimeError('Server failed to start (timeout).\n')

    def _initialize_kernels(self) -> None:

        print(f'Initializing {self.cfg.workers} persistent Jupyter kernels...')
        start_time = time.time()

        self.sandbox_pool = queue.Queue()

        def _create_sandbox():
            return AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)

        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            futures = [executor.submit(_create_sandbox) for _ in range(self.cfg.workers)]
            for future in as_completed(futures):
                self.sandbox_pool.put(future.result())

        elapsed = time.time() - start_time
        print(f'Kernels initialized in {elapsed:.2f} seconds.\n')

    def _scan_for_answer(self, text: str) -> int | None:
        """Hardened answer extraction — tries multiple patterns."""

        # 1. Primary: \boxed{N}
        matches = self.BOXED_RE.findall(text)
        if matches:
            try:
                value = int(matches[-1].replace(',', ''))
                if 0 <= value <= 99999:
                    return value
            except ValueError:
                pass

        # 2. Fallback: "the answer is N" / "answer: N"
        matches = self.ANSWER_IS_RE.findall(text)
        if matches:
            try:
                value = int(matches[-1])
                if 0 <= value <= 99999:
                    return value
            except ValueError:
                pass

        # 3. Fallback: "therefore N" / "thus N"
        matches = self.THEREFORE_RE.findall(text)
        if matches:
            try:
                value = int(matches[-1])
                if 0 <= value <= 99999:
                    return value
            except ValueError:
                pass

        return None

    def _scan_full_response(self, text: str) -> int | None:
        """Scan BOTH visible text and thinking block for answers.
        
        Gemma 4 sometimes puts the answer in the thinking channel
        before repeating it in the visible response. Check both.
        """
        # First try visible text
        visible = self._strip_thinking(text)
        answer = self._scan_for_answer(visible)
        if answer is not None:
            return answer

        # Then try the full text including thinking
        return self._scan_for_answer(text)

    def _strip_thinking(self, text: str) -> str:
        return self.THINK_RE.sub('', text).strip()

    def _extract_code_blocks(self, text: str) -> list[str]:
        visible = self._strip_thinking(text)
        return self.CODE_BLOCK_RE.findall(visible)

    def _clean_messages_for_api(self, messages: list[dict]) -> list[dict]:
        """Strip thinking from history per Gemma 4 best practices."""
        cleaned = []
        for msg in messages:
            if msg['role'] == 'assistant':
                cleaned_content = self._strip_thinking(msg['content'])
                if cleaned_content:
                    cleaned.append({**msg, 'content': cleaned_content})
            else:
                cleaned.append(msg)
        return cleaned

    def _process_attempt(
        self,
        problem: str,
        system_prompt: str,
        attempt_index: int,
        stop_event: threading.Event,
        deadline: float,
        enable_code: bool = True,
    ) -> dict:
        """Run a single branch.
        
        Args:
            enable_code: If True, include tool prompt and execute code blocks.
                        If False, pure CoT reasoning only (no code execution).
        """

        if stop_event.is_set() or time.time() > deadline:
            return {
                'Attempt': attempt_index + 1,
                'Family': 'TIR' if enable_code else 'CoT',
                'Answer': None,
                'Python Calls': 0,
                'Python Errors': 0,
                'Response Length': 0,
                'Trace': '',
            }

        local_tool = None
        sandbox = None
        python_calls = 0
        python_errors = 0
        total_tokens = 0
        final_answer = None
        full_trace = ''

        attempt_seed = int(math.pow(self.cfg.seed + attempt_index, 2))

        try:
            if enable_code:
                sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)
                local_tool = Gemma4Tool(
                    local_jupyter_timeout=self.cfg.jupyter_timeout,
                    tool_prompt=self.cfg.tool_prompt,
                    sandbox=sandbox
                )

            # Build messages — with or without tool instruction
            sys_content = system_prompt
            if enable_code and local_tool:
                sys_content += '\n\n' + local_tool.instruction

            messages = [
                {'role': 'system', 'content': sys_content},
                {'role': 'user', 'content': problem},
            ]

            for turn_idx in range(self.cfg.turns):
                if stop_event.is_set() or time.time() > deadline:
                    break

                api_messages = self._clean_messages_for_api(messages)
                remaining_time = max(1, deadline - time.time())

                try:
                    response = self.client.chat.completions.create(
                        model=self.cfg.served_model_name,
                        messages=api_messages,
                        temperature=self.cfg.temperature,
                        top_p=self.cfg.top_p,
                        max_tokens=8192,
                        seed=attempt_seed + turn_idx,
                        extra_body={'top_k': self.cfg.top_k},
                        timeout=remaining_time,
                    )
                except Exception as e:
                    break

                if not response.choices:
                    break

                reply_text = response.choices[0].message.content or ''
                total_tokens += response.usage.completion_tokens if response.usage else 0

                # Scan full response (including thinking) for answer
                answer_candidate = self._scan_full_response(reply_text)
                if answer_candidate is not None:
                    final_answer = answer_candidate

                visible_text = self._strip_thinking(reply_text)
                full_trace += visible_text + '\n'

                # Extract code blocks from visible text
                code_blocks = self._extract_code_blocks(reply_text) if enable_code else []

                if not code_blocks:
                    if final_answer is not None:
                        break
                    messages.append({'role': 'assistant', 'content': reply_text})
                    if enable_code:
                        messages.append({
                            'role': 'user',
                            'content': 'Please continue solving. Write Python code to compute and verify, then provide your answer as \\boxed{ANSWER}.'
                        })
                    else:
                        messages.append({
                            'role': 'user',
                            'content': 'Please continue your reasoning and provide your final answer as \\boxed{ANSWER}.'
                        })
                    continue

                messages.append({'role': 'assistant', 'content': reply_text})

                for code in code_blocks:
                    if stop_event.is_set() or time.time() > deadline:
                        break
                    python_calls += 1
                    result = local_tool.execute_code(code)
                    if result['is_error']:
                        python_errors += 1
                    output_text = result['output']
                    if len(output_text) > 3000:
                        output_text = output_text[:1500] + '\n...[truncated]...\n' + output_text[-1500:]
                    messages.append({
                        'role': 'user',
                        'content': f'Code execution result:\n```\n{output_text}\n```'
                    })
                    full_trace += f'[CODE]: {output_text[:300]}\n'

                if final_answer is not None:
                    break

        except Exception as exc:
            python_errors += 1

        finally:
            if local_tool is not None:
                local_tool.close()
            if sandbox is not None:
                sandbox.reset()
                self.sandbox_pool.put(sandbox)

        return {
            'Attempt': attempt_index + 1,
            'Family': 'TIR' if enable_code else 'CoT',
            'Response Length': total_tokens,
            'Python Calls': python_calls,
            'Python Errors': python_errors,
            'Answer': final_answer,
            'Trace': full_trace[:2000],
        }

    def _genselect(self, problem: str, detailed_results: list, top_n: int = 3) -> int | None:
        """GenSelect: ask the model to judge between candidate solutions.
        
        Adapted for Gemma 4:
        - Uses low temperature (0.3) for precise judging
        - Feeds truncated solution traces for context
        - Multiple selector passes with majority vote among selectors
        """

        answer_counts = Counter()
        answer_traces = defaultdict(list)
        for r in detailed_results:
            if r['Answer'] is not None:
                answer_counts[r['Answer']] += 1
                if r.get('Trace'):
                    answer_traces[r['Answer']].append(r['Trace'])

        if len(answer_counts) < 2:
            return None

        top_candidates = answer_counts.most_common(top_n)

        # Build GenSelect prompt
        candidates_text = ''
        for i, (ans, votes) in enumerate(top_candidates):
            traces = answer_traces.get(ans, [])
            best_trace = max(traces, key=len) if traces else 'No trace available.'
            if len(best_trace) > 1200:
                best_trace = best_trace[:600] + '\n...[truncated]...\n' + best_trace[-600:]
            candidates_text += f'\n\n--- Solution {i+1} (claims answer = {ans}, supported by {votes} branches) ---\n{best_trace}'

        genselect_messages = [
            {
                'role': 'system',
                'content': (
                    '<|think|>'
                    'You are a mathematical expert and judge for the International Mathematical Olympiad. '
                    'You will be given a math problem and several candidate solutions with their claimed answers. '
                    'Carefully analyze each solution for: '
                    '1) Logical errors or unjustified steps. '
                    '2) Computational mistakes. '
                    '3) Whether the approach actually solves the stated problem. '
                    '4) Whether the final answer follows from the reasoning. '
                    'After your analysis, output ONLY the correct integer answer inside \\boxed{}.'
                )
            },
            {
                'role': 'user',
                'content': f'Problem:\n{problem}\n\nCandidate Solutions:{candidates_text}\n\nAnalyze the solutions and give the correct answer as \\boxed{{ANSWER}}.'
            }
        ]

        selector_answers = []
        for i in range(self.cfg.genselect_passes):
            try:
                response = self.client.chat.completions.create(
                    model=self.cfg.served_model_name,
                    messages=genselect_messages,
                    temperature=0.3,
                    top_p=0.9,
                    max_tokens=256,
                    seed=self.cfg.seed + 2000 + i,
                    extra_body={'top_k': self.cfg.top_k},
                    timeout=45,
                )
                if response.choices:
                    text = response.choices[0].message.content or ''
                    ans = self._scan_full_response(text)
                    if ans is not None:
                        selector_answers.append(ans)
            except Exception:
                pass

        if not selector_answers:
            return None

        return Counter(selector_answers).most_common(1)[0][0]

    def _select_answer(self, detailed_results: list) -> int:
        """Majority vote with python-call weighting (same as baseline)."""

        stats = defaultdict(lambda: {'votes': 0, 'calls': 0})
        for result in detailed_results:
            answer = result['Answer']
            if answer is not None:
                stats[answer]['votes'] += 1
                stats[answer]['calls'] += result['Python Calls']

        sorted_stats = sorted(
            stats.items(),
            key=lambda item: (item[1]['votes'], item[1]['calls']),
            reverse=True
        )

        vote_data = []
        for answer, data in sorted_stats:
            vote_data.append((answer, data['votes'], data['calls']))

        vote_dataframe = pd.DataFrame(vote_data, columns=['Answer', 'Votes', 'Calls'])
        display(vote_dataframe)

        final_answer = sorted_stats[0][0]
        final_votes = sorted_stats[0][1]['votes']
        final_calls = sorted_stats[0][1]['calls']

        print(f'\nMajority Vote: {final_answer} | Votes: {final_votes} | Calls: {final_calls}\n')

        return final_answer

    def solve_problem(self, problem: str) -> int:

        problem_start_time = time.time()
        print(f'\nProblem: {problem[:200]}...\n')

        user_input = f'{problem} {self.cfg.preference_prompt}'
        elapsed_global = time.time() - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed_global
        problems_left_others = max(0, self.problems_remaining - 1)
        reserved_time = problems_left_others * self.cfg.base_problem_timeout

        budget = time_left - reserved_time
        budget = min(budget, self.cfg.high_problem_timeout)
        budget = max(budget, self.cfg.base_problem_timeout)

        deadline = time.time() + budget
        print(f'Budget: {budget:.2f} seconds\n')

        # ── Phase 1: Mixed branch families ──
        # TIR branches (code-enabled) + CoT branches (pure reasoning)
        n_tir = self.cfg.n_tir_branches
        n_cot = self.cfg.n_cot_branches

        detailed_results = []
        valid_answers = []
        stop_event = threading.Event()
        executor = ThreadPoolExecutor(max_workers=self.cfg.workers)

        try:
            futures = []
            
            # Launch TIR branches
            for i in range(n_tir):
                future = executor.submit(
                    self._process_attempt,
                    user_input,
                    self.cfg.system_prompt,
                    i,
                    stop_event,
                    deadline,
                    enable_code=True,
                )
                futures.append(future)

            # Launch CoT branches (no code)
            for i in range(n_cot):
                future = executor.submit(
                    self._process_attempt,
                    user_input,
                    self.cfg.system_prompt,
                    i + 50,  # Different seed range
                    stop_event,
                    deadline,
                    enable_code=False,
                )
                futures.append(future)

            for future in as_completed(futures):
                try:
                    result = future.result()
                    detailed_results.append(result)
                    if result['Answer'] is not None:
                        valid_answers.append(result['Answer'])
                    counts = Counter(valid_answers).most_common(1)
                    if counts and counts[0][1] >= self.cfg.early_stop:
                        stop_event.set()
                        for f in futures:
                            f.cancel()
                        break
                except Exception as exc:
                    print(f'Future failed: {exc}')
                    continue
        finally:
            executor.shutdown(wait=False, cancel_futures=True)

        # ── Phase 2: Rescue branches on weak consensus ──
        counts = Counter(valid_answers).most_common()
        top_votes = counts[0][1] if counts else 0
        remaining_time = deadline - time.time()
        used_time = time.time() - problem_start_time
        is_tied = len(counts) >= 2 and counts[0][1] == counts[1][1]
        needs_rescue = (top_votes <= 2 or is_tied) and valid_answers
        has_time = remaining_time > used_time

        if needs_rescue and has_time and valid_answers:
            n_extra = self.cfg.rescue_branches
            print(f'\n[Rescue] Weak consensus ({top_votes} votes). Running {n_extra} extra TIR branches... ({remaining_time:.0f}s left)\n')
            extra_stop = threading.Event()
            extra_executor = ThreadPoolExecutor(max_workers=self.cfg.workers)
            try:
                extra_futures = []
                for i in range(n_extra):
                    future = extra_executor.submit(
                        self._process_attempt,
                        user_input,
                        self.cfg.system_prompt,
                        i + 100,
                        extra_stop,
                        deadline,
                        enable_code=True,
                    )
                    extra_futures.append(future)
                for future in as_completed(extra_futures):
                    try:
                        result = future.result()
                        detailed_results.append(result)
                        if result['Answer'] is not None:
                            valid_answers.append(result['Answer'])
                    except Exception:
                        pass
            finally:
                extra_stop.set()
                extra_executor.shutdown(wait=False, cancel_futures=True)

        # ── Phase 3: GenSelect ──
        genselect_answer = None
        genselect_used = False
        counts = Counter(valid_answers).most_common()
        remaining_time = deadline - time.time()
        n_distinct = len(counts)
        is_unanimous = n_distinct == 1 and len(valid_answers) >= 3

        if (self.cfg.use_genselect
            and not is_unanimous
            and n_distinct >= 2
            and remaining_time > 45
            and valid_answers):

            print(f'\n[GenSelect] {n_distinct} distinct answers. Running GenSelect... ({remaining_time:.0f}s left)\n')
            genselect_answer = self._genselect(problem, detailed_results, top_n=min(n_distinct, 3))
            genselect_used = True
            if genselect_answer is not None:
                print(f'  GenSelect picked: {genselect_answer}')
            else:
                print(f'  GenSelect returned no answer — falling back to majority vote')

        # ── Logging and result ──
        self.problems_remaining = max(0, self.problems_remaining - 1)
        used_time = time.time() - problem_start_time
        saved_time = max(0.0, budget - used_time)

        # Count by family
        tir_results = [r for r in detailed_results if r.get('Family') == 'TIR']
        cot_results = [r for r in detailed_results if r.get('Family') == 'CoT']
        tir_answered = sum(1 for r in tir_results if r['Answer'] is not None)
        cot_answered = sum(1 for r in cot_results if r['Answer'] is not None)

        print(f"\n[Budget]: {budget:.2f}s")
        print(f"[Inference] Took {used_time:.2f}s")
        print(f"[Saved]: {saved_time:.2f}s")
        print(f"[Branches] TIR: {len(tir_results)} ({tir_answered} answered) | CoT: {len(cot_results)} ({cot_answered} answered)")
        if genselect_used:
            print(f"[GenSelect] {'Override' if genselect_answer else 'No result'}")
        print()

        if detailed_results:
            results_dataframe = pd.DataFrame([
                {k: v for k, v in r.items() if k != 'Trace'}
                for r in detailed_results
            ])
            results_dataframe['Answer'] = results_dataframe['Answer'].astype('Int64')
            display(results_dataframe)

        if not valid_answers:
            print('\nResult: 0\n')
            self.problem_logs.append({'time': used_time, 'answer': 0, 'genselect': False, 'branches': len(detailed_results)})
            return 0

        try:
            import requests
            requests.post(f'{self.base_url}/reset_prefix_cache')
        except Exception:
            pass

        # Final answer: GenSelect if available, else majority vote
        mv_answer = self._select_answer(detailed_results)

        if genselect_answer is not None:
            # If GenSelect agrees with majority → high confidence
            if genselect_answer == mv_answer:
                print(f'🎯 GenSelect confirms majority vote: {mv_answer}\n')
                final = mv_answer
            else:
                # GenSelect disagrees — trust GenSelect on close votes,
                # trust majority on strong consensus
                top_votes = counts[0][1] if counts else 0
                total_valid = len(valid_answers)
                majority_pct = top_votes / max(total_valid, 1)

                if majority_pct >= 0.6:
                    print(f'⚖️ Strong majority ({majority_pct:.0%}) overrides GenSelect. Using: {mv_answer}\n')
                    final = mv_answer
                else:
                    print(f'🧠 Weak majority ({majority_pct:.0%}). Trusting GenSelect: {genselect_answer}\n')
                    final = genselect_answer
        else:
            final = mv_answer

        self.problem_logs.append({
            'time': used_time,
            'answer': final,
            'genselect': genselect_used,
            'genselect_override': genselect_answer is not None and genselect_answer != mv_answer,
            'branches': len(detailed_results),
            'tir_branches': len(tir_results),
            'cot_branches': len(cot_results),
            'distinct_answers': len(counts),
            'top_votes': counts[0][1] if counts else 0,
        })

        return final

    def __del__(self):
        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()
        if hasattr(self, 'log_file'):
            self.log_file.close()
        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                try:
                    sb = self.sandbox_pool.get_nowait()
                    sb.close()
                except Exception:
                    pass


In [ ]:
solver = Gemma4AdaptedSolver(CFG)

In [ ]:
TEST_MODE = True   # <-- Set to False for submission
N_TEST =   10      # Number of problems to test

if TEST_MODE:
    import time
    
    ref_path = None
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if f == 'reference.csv':
                ref_path = os.path.join(root, f)
                break
    
    if ref_path is None:
        print("reference.csv not found!")
    else:
        ref_df = pl.read_csv(ref_path)
        print(f"Loaded {len(ref_df)} problems from {ref_path}")
        
        correct = 0
        total = 0
        
        for i in range(min(N_TEST, len(ref_df))):
            row = ref_df.row(i, named=True)
            question = row.get('question', row.get('problem', ''))
            expected = row.get('answer', None)
            
            print(f"\n{'='*60}")
            print(f"Problem {i+1}/{N_TEST}: {question[:150]}...")
            
            start = time.time()
            predicted = solver.solve_problem(question)
            elapsed = time.time() - start
            
            match = "✓" if predicted == expected else "✗"
            if predicted == expected:
                correct += 1
            total += 1
            
            print(f"Predicted: {predicted} | Expected: {expected} | {match} | {elapsed:.1f}s")
        
        print(f"\n{'='*60}")
        print(f"SCORE: {correct}/{total} correct")

In [ ]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    global correct_count, total_count, predictions
    
    question_id = id_.item(0)
    question_text = question.item(0)
    
    print("------")
    print(f"ID: {question_id}")
    print(f"Question: {question_text[:200]}...")
    
    final_answer = solver.solve_problem(question_text)
    predictions[question_id] = final_answer

    # Check accuracy if ground truth available
    total_count += 1
    if question_id in ground_truth:
        gt = ground_truth[question_id]
        is_correct = (final_answer == gt)
        if is_correct:
            correct_count += 1
        status = "✅" if is_correct else "❌"
        print(f"Answer: {final_answer} | Ground Truth: {gt} | {status}")
        print(f"📊 Running Accuracy: {correct_count}/{total_count} ({100*correct_count/total_count:.1f}%)")
    else:
        print(f"Answer: {final_answer}")
    
    print("------\n")
    
    return pl.DataFrame({'id': question_id, 'answer': final_answer})

In [ ]:
# Load reference data and keep ground truth for accuracy calculation
df = pd.read_csv(
    "/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv"
)

# Store ground truth answers for accuracy calculation (only in local mode)
ground_truth = dict(zip(df["id"], df["answer"])) if "answer" in df.columns else {}

# Create input file without answers
df.drop("answer", axis=1, errors="ignore").to_csv("reference.csv", index=False)

# Track predictions for accuracy calculation
predictions = {}
correct_count = 0
total_count = 0

In [ ]:
# ============================================================
# ABLATION STUDY & METRICS ANALYSIS
# Run this cell after inference completes to get detailed stats
# ============================================================

import json
from collections import defaultdict

def run_ablation(predictions, ground_truth, solver_logs=None):
    """Comprehensive ablation analysis."""
    
    print("=" * 70)
    print("ABLATION STUDY RESULTS")
    print("=" * 70)
    
    # 1. Overall accuracy
    total = len(predictions)
    correct = sum(1 for pid, ans in predictions.items() if ground_truth.get(pid) == ans)
    wrong = sum(1 for pid, ans in predictions.items() if pid in ground_truth and ground_truth[pid] != ans)
    no_answer = sum(1 for pid, ans in predictions.items() if ans == 0 or ans is None)
    
    print(f"\n📊 OVERALL ACCURACY: {correct}/{total} ({100*correct/max(total,1):.1f}%)")
    print(f"   Correct:   {correct}")
    print(f"   Wrong:     {wrong}")  
    print(f"   No answer: {no_answer}")
    
    # 2. Per-problem breakdown
    print(f"\n{'='*70}")
    print("PER-PROBLEM BREAKDOWN")
    print(f"{'='*70}")
    print(f"{'ID':>6} | {'Predicted':>10} | {'Truth':>10} | {'Status':>8}")
    print("-" * 50)
    
    problems_correct = []
    problems_wrong = []
    
    for pid in sorted(predictions.keys()):
        pred = predictions[pid]
        gt = ground_truth.get(pid, '?')
        if gt == '?':
            status = '❓'
        elif pred == gt:
            status = '✅'
            problems_correct.append(pid)
        else:
            status = '❌'
            problems_wrong.append(pid)
        print(f"{pid:>6} | {pred:>10} | {str(gt):>10} | {status:>8}")
    
    # 3. Timing analysis (if logs available)
    if solver_logs:
        print(f"\n{'='*70}")
        print("TIMING ANALYSIS")
        print(f"{'='*70}")
        
        times = [log.get('time', 0) for log in solver_logs]
        if times:
            print(f"   Mean time per problem: {sum(times)/len(times):.1f}s")
            print(f"   Median:               {sorted(times)[len(times)//2]:.1f}s")
            print(f"   Min:                  {min(times):.1f}s")
            print(f"   Max:                  {max(times):.1f}s")
            print(f"   Total:                {sum(times):.1f}s ({sum(times)/3600:.2f}h)")
    
    # 4. Confidence distribution  
    print(f"\n{'='*70}")
    print("WHAT TO COMPARE BETWEEN NOTEBOOKS")
    print(f"{'='*70}")
    print("""
    Metric                    | Notebook A (high-branch) | Notebook B (baseline+GS)
    --------------------------+--------------------------+-------------------------
    Score (correct/50)        |                          |
    Time per problem (mean)   |                          |
    Time per problem (median) |                          |
    Total inference time      |                          |
    Branches per problem      |                          |
    Early stops triggered     |                          |
    Rescue branches triggered |                          |
    GenSelect triggered       |                          |
    GenSelect override rate   |                          |
    GenSelect accuracy        |                          |
    Problems with 0 answer    |                          |
    Problems with tied votes  |                          |
    Avg votes for winner      |                          |
    
    Fill in after running both notebooks on the same reference set.
    The notebook with the higher score AND lower time is the winner.
    If scores are equal, pick the one with lower time (more headroom).
    """)
    
    return {
        'total': total,
        'correct': correct,
        'wrong': wrong,
        'no_answer': no_answer,
        'accuracy': correct / max(total, 1),
        'problems_correct': problems_correct,
        'problems_wrong': problems_wrong,
    }


# === FP8 vs BF16 COMPARISON GUIDE ===
print("""
╔══════════════════════════════════════════════════════════════════════╗
║  FP8 vs BF16 WEIGHT COMPARISON — HOW TO TEST                      ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  To test FP8 weights, change CFG.dtype from 'bfloat16' to 'auto'  ║
║  and add to vLLM cmd: '--quantization', 'fp8'                     ║
║                                                                    ║
║  WARNING: FP8 runtime quant for Gemma 4 MoE is BUGGY in vLLM.    ║
║  If it crashes, FP8 is not viable and BF16 is the answer.         ║
║                                                                    ║
║  If it works, compare:                                             ║
║    BF16: score=??, time=??, branches=12                            ║
║    FP8:  score=??, time=??, branches=20+ (more VRAM for KV)       ║
║                                                                    ║
║  If FP8 score >= BF16 score - 1: use FP8 (more branches wins)    ║
║  If FP8 score <  BF16 score - 1: use BF16 (quality wins)         ║
╚══════════════════════════════════════════════════════════════════════╝
""")

# Run ablation if data is available
try:
    if predictions and ground_truth:
        ablation_results = run_ablation(predictions, ground_truth)
except NameError:
    print("Run inference first, then re-run this cell for ablation analysis.")


In [ ]:
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
    
else:
    inference_server.run_local_gateway(("reference.csv",))
    #inference_server.run_local_gateway(
    #    ('/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv',)
    #)